# Train an image classifier for Local AI Lab (PyTorch → ONNX)

Trains on the **training images** of a Model Arena image dataset (a ZIP of class folders), exports ONNX with the
metadata Local AI Lab understands (labels in output order, preprocessing, dataset and split hashes), and gives you a
file to upload under **Model Arena → Models**.

> **Privacy first.** Google Colab runs on Google's cloud. Only upload data you are allowed to process there –
> **never confidential documents, patient, customer or employee data** unless your organisation approved it.
> The same notebook runs on your own computer with Jupyter (`pip install notebook`, then `jupyter notebook`), which
> keeps everything offline. Using Colab is subject to Google's terms; the libraries used here are open source
> (licences listed at the end).

**Weights and licences.** `PRETRAINED = False` trains a small CNN from scratch – no third-party weights involved.
`PRETRAINED = True` starts from torchvision's ImageNet weights (much better with few images). torchvision's code is
BSD-3-Clause, but *pre-trained weights may carry their own terms derived from their training data* (ImageNet's
terms restrict commercial use); check https://pytorch.org/vision/stable/models.html before using such a model
commercially. Use a GPU runtime in Colab (*Runtime → Change runtime type → T4 GPU*).

**You need from Local AI Lab:** the image ZIP you uploaded and *Model Arena → your dataset → Download split*.

In [ ]:
# 1. Install (Colab already has torch / torchvision)
%pip install -q "onnx>=1.16" "onnxruntime>=1.18" pillow

In [ ]:
import io, os, json, zipfile
ZIP_PATH, SPLIT_PATH = "images.zip", "split.csv"
try:
    from google.colab import files
    up = files.upload()
    ZIP_PATH = next(n for n in up if n.lower().endswith(".zip"))
    SPLIT_PATH = next(n for n in up if "split" in n.lower())
except ImportError:
    pass

PRETRAINED = False   # True = torchvision MobileNetV3-Small ImageNet weights (see licence note above)
IMAGE_SIZE = 224 if PRETRAINED else 96
EPOCHS = 10
BATCH = 32

In [ ]:
import pandas as pd
with open(SPLIT_PATH, encoding="utf-8") as f:
    header = f.readline()
meta = dict(kv.split("=", 1) for kv in header.lstrip("# ").split() if "=" in kv)
split = pd.read_csv(SPLIT_PATH, skiprows=1)  # first line: hashes
DATASET_SHA256, SPLIT_SHA256 = meta.get("dataset_sha256"), meta.get("split_sha256")
member = dict(zip(split["source"], split["split"]))

SPLIT_DIRS = {"train", "training", "test", "testing", "val", "valid", "validation", "dev"}
items = []  # (zip name, label, split)
with zipfile.ZipFile(ZIP_PATH) as zf:
    for info in zf.infolist():
        name = info.filename.replace("\\", "/")
        if info.is_dir() or "__MACOSX" in name or name.split("/")[-1].startswith("."):
            continue
        parts = name.split("/")
        for k in range(len(parts)):  # the split file lists the path inside the ZIP (maybe without a wrapper folder)
            src = "/".join(parts[k:])
            if src in member:
                items.append((info.filename, parts[-2], member[src]))
                break
LABELS = sorted({lab for _, lab, _ in items})   # Local AI Lab orders classes alphabetically
train_items = [(n, LABELS.index(l)) for n, l, s in items if s == "train"]
print(len(items), "images,", len(train_items), "for training – validation/test images are NOT used. Classes:", LABELS)

In [ ]:
import random
import torch
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

device = "cuda" if torch.cuda.is_available() else "cpu"
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]   # = "imagenet" normalisation in Local AI Lab
train_tf = transforms.Compose([transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)), transforms.RandomHorizontalFlip(),
                               transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
eval_tf = transforms.Compose([transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)), transforms.ToTensor(),
                              transforms.Normalize(MEAN, STD)])

class ZipImages(Dataset):
    def __init__(self, entries, tf):
        self.entries, self.tf, self.zf = entries, tf, None
    def __len__(self):
        return len(self.entries)
    def __getitem__(self, i):
        if self.zf is None:
            self.zf = zipfile.ZipFile(ZIP_PATH)
        name, y = self.entries[i]
        img = Image.open(io.BytesIO(self.zf.read(name))).convert("RGB")
        return self.tf(img), y

random.seed(0)
random.shuffle(train_items)
hold = max(1, len(train_items) // 10)   # early stopping uses a slice of the TRAINING images only
fit_items, hold_items = train_items[hold:], train_items[:hold]
fit_dl = DataLoader(ZipImages(fit_items, train_tf), batch_size=BATCH, shuffle=True, num_workers=2)
hold_dl = DataLoader(ZipImages(hold_items, eval_tf), batch_size=BATCH)

if PRETRAINED:
    from torchvision.models import MobileNet_V3_Small_Weights, mobilenet_v3_small
    model = mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.DEFAULT)
    model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, len(LABELS))
else:
    def block(i, o):
        return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(), nn.MaxPool2d(2))
    model = nn.Sequential(block(3, 32), block(32, 64), block(64, 128), block(128, 128),
                          nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.2), nn.Linear(128, len(LABELS)))
model = model.to(device)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3 if not PRETRAINED else 3e-4)
loss_fn = nn.CrossEntropyLoss()

In [ ]:
best, best_state = -1.0, None
for epoch in range(EPOCHS):
    model.train()
    for xb, yb in fit_dl:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward()
        opt.step()
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for xb, yb in hold_dl:
            pred = model(xb.to(device)).argmax(1).cpu()
            correct += (pred == yb).sum().item()
            total += len(yb)
    acc = correct / max(1, total)
    print(f"epoch {epoch + 1}: hold-out (from training images) accuracy {acc:.3f}")
    if acc > best:
        best, best_state = acc, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
model.load_state_dict(best_state)

## Export to ONNX with Local AI Lab metadata

In [ ]:
import onnx
model = model.cpu().eval()
dummy = torch.zeros(1, 3, IMAGE_SIZE, IMAGE_SIZE)
OUT = "image_model.onnx"
torch.onnx.export(model, dummy, OUT, input_names=["input"], output_names=["logits"],
                  dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}}, opset_version=17)
m = onnx.load(OUT)
meta = {"kind": "image", "task": "classification", "labels": json.dumps(LABELS),
        "preprocess": json.dumps({"size": IMAGE_SIZE, "layout": "nchw", "normalization": "imagenet", "color": "rgb",
                                  "resize_mode": "resize", "channels": 3}),
        "dataset_sha256": DATASET_SHA256, "split_sha256": SPLIT_SHA256, "trained_on": "train",
        "framework": f"pytorch {torch.__version__}",
        "weights": "torchvision MobileNetV3-Small ImageNet (check licence)" if PRETRAINED else "trained from scratch",
        "notebook": "local-ai-lab/notebooks/train_image_classifier_pytorch.ipynb"}
for k, v in meta.items():
    if v is not None:
        p = m.metadata_props.add()
        p.key, p.value = f"local_ai_lab.{k}", str(v)
onnx.save(m, OUT)
import onnxruntime as ort
print(ort.InferenceSession(OUT).run(None, {"input": dummy.numpy()})[0].shape, f"{os.path.getsize(OUT) / 1e6:.1f} MB")
try:
    from google.colab import files
    files.download(OUT)
except ImportError:
    print("model written to", os.path.abspath(OUT))

## Use it in Local AI Lab
**Model Arena → Models → Upload model** → `image_model.onnx`, data type *image*, task *classification*. Labels and
preprocessing are filled in from the file; training data shows *trained only on this dataset's training split*.
Then compare it with the built-in classifiers (raw pixels, MobileNetV2 / EfficientNet-Lite features) on the same
dataset. Because this model never saw the validation/test images, the comparison is fair.

You can also upload it with role **feature extractor**: Local AI Lab then trains small classifiers on its features.

**Licences:** PyTorch / torchvision (BSD-3-Clause), ONNX (Apache-2.0), ONNX Runtime (MIT), Pillow (MIT-CMU). Pre-trained
weights: see the note at the top.